In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [2]:
#Load the trained model
model=load_model('model.h5')

#Load the scaler and encoder
with open('scaler.pkl', 'rb') as file:
    scaler=pickle.load(file)
with open('label_encode_gender.pkl', 'rb') as file:
    label_encode_gender=pickle.load(file)
with open('onehot_encoder_geo.pkl', 'rb') as file:
    onehot_encoder_geo=pickle.load(file)


In [3]:
## input data
#CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
input_data={
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}


In [4]:
#Preprocess the input data
#Create a DataFrame from the input data
input_df=pd.DataFrame([input_data])

#Encode the categorical features
input_df['Gender']=label_encode_gender.transform(input_df['Gender'])
#One-hot encode the 'Geography' feature
geo_encoded=onehot_encoder_geo.transform(input_df[['Geography']]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
#Drop the original 'Geography' column and concatenate the one-hot encoded columns
input_df=input_df.drop('Geography', axis=1)
input_df=pd.concat([input_df, geo_encoded_df], axis=1)

input_df


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [5]:
#scale the input data
input_scaled=scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [6]:
# predict the output
prediction=model.predict(input_scaled)
print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
[[0.02130138]]


In [7]:
prediction_prob=prediction[0][0]
prediction_prob

0.021301383

In [8]:
if prediction_prob>0.5:
    print("The customer is likely to exit.")
else:
    print("The customer is likely to stay.")

The customer is likely to stay.
